In [1]:
import sys
from absl import flags
from ml_collections.config_flags import config_flags

sys.argv = ["",
            "--config=td_sa_stack/config.py"]

config_flags.DEFINE_config_file("config", None, "Training configuration.", lock_config=True)

FLAGS = flags.FLAGS
FLAGS(sys.argv)

config = FLAGS.config

In [2]:
config.data

gamma: 0.9
max_traj_len: 25
min_traj_len: 5
num_threads: 48
random_flip: true
reward_final: 10
uniform_dequantization: true
with_random_actions: true
with_reversed_actions: false

In [3]:
import tensorflow as tf
tf.config.experimental.set_visible_devices([], "GPU")

import os
import jax
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.95'

from td_sa_stack import get_dataset, TrainerModule, RegressionInceptionNetV1

%load_ext autoreload
%autoreload 2

2025-06-15 20:56:34.236960: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750020994.256411    8377 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750020994.262748    8377 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750020994.278363    8377 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750020994.278379    8377 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750020994.278381    8377 computation_placer.cc:177] computation placer alr

In [4]:
!nvidia-smi

Sun Jun 15 20:56:41 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-16GB           On  |   00000000:3B:00.0 Off |                    0 |
| N/A   41C    P0             55W /  300W |    2387MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
jax.devices()

[CudaDevice(id=0)]

In [5]:
train_ds, _, _ = get_dataset(config, uniform_dequantization=config.data.uniform_dequantization)

trainer = TrainerModule(config=config,
                        model_class=RegressionInceptionNetV1,
                        version=14)

Batch dimensions: [1, 32]
Initializing model with batch shape: (32, 32, 32, 6)


wandb: Currently logged in as: shapka-pa (shapka-pa-moscow-institute-of-physics-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


In [6]:
try:
	trainer.train_model(train_ds=train_ds)
except KeyboardInterrupt as e:
	print(f"Обучение прервано")

2025-06-15 20:15:29.995610: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


KeyboardInterrupt: 